In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB")
        print(f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n")
else:
    print("No CUDA-compatible GPUs found.")

PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
Total GPUs Detected: 1

--- GPU 0 ---
  Name: NVIDIA H200
  Compute Capability: 9.0
  Total VRAM: 139.80 GB
  Allocated VRAM: 0.00 GB
  Free VRAM: 139.80 GB



In [4]:
notebook_login()

In [5]:
MODEL_ID = "XiaomiMiMo/MiMo-V2.6-Distill-Qwen-9B"
HF_USER = "Vishva007"               # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 17 files: 100%|██████████████████████| 17/17 [00:00<00:00, 465.72it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [8]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1000,                                 # Optimization iterations per block; high accuracy (Production grade)
    group_size=16,                              # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,                                   # Symmetric quantization; enables faster compute and broader engine compatibility
    gradient_accumulate_steps=4,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme="NVFP4",                             # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,                               # Batch size for calibration; keep < 8 to avoid slow execution/high memory overhead
    seqlen=2048,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,                  # Compiles forward passes using torch.compile to accelerate calibration (JIT acceleration)
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision (Crucial for VLM accuracy)
    ignore_layers="mtp,visual",                 # Excludes MTP, and visual modules from quantization (Add linear-attention if needed)
    layer_config={
        "mtp": {"data_type": "bfloat16"},       # Keeps the MTP module in BF16 precision
        "mtp.fc": {"data_type": "bfloat16"},    # Keeps the MTP fully connected layer in BF16 precision
    },                                          # Preserves BF16 precision for the specified MTP layers
)

In [9]:
# Quantize once and export to disk using the recommended schema for NVFP4
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Use 'llm_compressor' for the compressed-tensors schema, recommended for NVFP4 inference with vLLM / Neural Magic
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

2026-10-25 16:25:15 INFO layer_config_resolver.py L408: Ignored layers: model.visual.blocks.[0-26].attn.proj, model.visual.blocks.[0-26].attn.qkv, model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp, visual
2026-10-25 16:25:15 WARNING logging.py L340: Layer name or regex 'mtp' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-10-25 16:25:15 WARNING logging.py L340: Layer name or regex 'mtp.fc' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-10-25 16:25:15 INFO layer_config_resolver.py L408: Ignored layers: model.visual.blocks.[0-26].attn.proj, model.visual.blocks.[0-26].attn.qkv, model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2, model.visual.merger.linear_fc1, model.visual.merger.linear_fc2, mtp

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1152)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-26): 27 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1152, out_features=3456, bias=True)
             (proj): Linear(in_features=1152, out_features=1152, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
             (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
             (act_fn): 

In [10]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Update model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [11]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [12]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"),
        f"{base_name}-NVFP4",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-nvfp-w4g16 to Vishva007/MiMo-V2.6-Distill-Qwen-9B-NVFP4...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/MiMo-V2.6-Distill-Qwen-9B-NVFP4
